In [ ]:
# PHẦN 1: CÀI ĐẶT & FIX LỖI "UNPICKLING" (PHIÊN BẢN ULTIMATE)

import os
import shutil
import json
import torch
import gc
import time
import nltk
import functools
import numpy as np
from google.colab import drive
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from datasets import Dataset

# 1. Cấp quyền cho các hàm Numpy
try:
    torch.serialization.add_safe_globals([
        np.core.multiarray._reconstruct,
        np.ndarray,
        np.dtype,
    ])
    print("✅ Đã whitelist Numpy cho PyTorch.")
except AttributeError:
    pass

if not hasattr(torch, 'original_load_func'):
    torch.original_load_func = torch.load

def safe_load_override(*args, **kwargs):
    if 'weights_only' in kwargs:
        del kwargs['weights_only']

    return torch.original_load_func(*args, weights_only=False, **kwargs)

torch.load = safe_load_override
print(f"✅ Đã ép buộc torch.load(weights_only=False) thành công.")


# --- KẾT NỐI DRIVE & KAGGLE ---
os.environ['KAGGLE_USERNAME'] = "phankhaclap"
os.environ['KAGGLE_KEY'] = "0ba946628cb1f5acb76ecd357f590e95"

drive.mount('/content/drive')

FINAL_SAVE_PATH = "/content/drive/My Drive/T5_Small_Spider_CRP_FFT"
CHECKPOINT_DIR = "/content/drive/My Drive/T5_Small_Spider_CRP_FFT/checkpoints"

print(">>> [1/5] Tải dữ liệu...")
if os.path.exists('spider_data'): shutil.rmtree('spider_data')
!pip install -q kaggle
!kaggle datasets download -d jeromeblanchet/yale-universitys-spider-10-nlp-dataset
!unzip -q yale-universitys-spider-10-nlp-dataset.zip -d temp_spider

if os.path.exists("temp_spider/spider"):
    shutil.move("temp_spider/spider", "spider_data")
    shutil.rmtree('temp_spider')
    !wget -q https://raw.githubusercontent.com/taoyds/spider/master/evaluation.py
    !wget -q https://raw.githubusercontent.com/taoyds/spider/master/process_sql.py
    nltk.download('punkt')
    nltk.download('punkt_tab')


# PHẦN 2: TIỀN XỬ LÝ

MODEL_NAME = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

with open('spider_data/tables.json', 'r') as f:
    table_data = json.load(f)
schema_map = {db['db_id']: db for db in table_data}

def get_crp_schema(db_id):
    if db_id not in schema_map: return ""
    db = schema_map[db_id]
    ddl_statements = []
    for i, table_name in enumerate(db['table_names_original']):
        col_defs = [f"{c[1]} {db['column_types'][j]}" for j, c in enumerate(db['column_names_original']) if c[0] == i]
        pk_idx = db['primary_keys']
        pks = [db['column_names_original'][j][1] for j in pk_idx if db['column_names_original'][j][0] == i]
        if pks: col_defs.append(f"primary key ({', '.join(pks)})")
        for fk in db['foreign_keys']:
            if db['column_names_original'][fk[0]][0] == i:
                from_col = db['column_names_original'][fk[1]][1]
                to_table = db['table_names_original'][db['column_names_original'][fk[1]][0]]
                to_col = db['column_names_original'][fk[1]][1]
                col_defs.append(f"foreign key ({from_col}) references {to_table}({to_col})")
        ddl_statements.append(f"CREATE TABLE {table_name} ({', '.join(col_defs)});")
    return " ".join(ddl_statements)

def load_spider_unified(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return Dataset.from_dict({
        "question": [item["question"] for item in data],
        "query": [item["query"] for item in data],
        "db_id": [item["db_id"] for item in data]
    })

def preprocess_fn(examples):
    inputs = [f"translate to SQL: {q} | Schema: {get_crp_schema(d)}" for q, d in zip(examples['question'], examples['db_id'])]
    model_inputs = tokenizer(inputs, max_length=512, padding="max_length", truncation=True)
    labels = tokenizer(examples['query'], max_length=128, padding="max_length", truncation=True)
    labels["input_ids"] = [[(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = load_spider_unified("spider_data/train_spider.json").map(preprocess_fn, batched=True)
val_ds = load_spider_unified("spider_data/dev.json").map(preprocess_fn, batched=True)

# PHẦN 3: HUẤN LUYỆN (RESUME)

print(f"\n>>> [3/5] Bắt đầu huấn luyện...")

last_checkpoint = None
if os.path.isdir(CHECKPOINT_DIR):
    checkpoints = [os.path.join(CHECKPOINT_DIR, d) for d in os.listdir(CHECKPOINT_DIR) if d.startswith("checkpoint-")]
    if len(checkpoints) > 0:
        checkpoints.sort(key=lambda x: int(x.split('-')[-1]))
        last_checkpoint = checkpoints[-1]
        print(f"Tìm thấy checkpoint cũ: {last_checkpoint}")

model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
model.gradient_checkpointing_enable()

training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=15,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    optim="adafactor",
    weight_decay=0.01,
    fp16=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    predict_with_generate=True,
    report_to="none",
    logging_steps=50,
    load_best_model_at_end=False
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds
)

if last_checkpoint:
    print(f"Tiếp tục train từ checkpoint: {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Bắt đầu train mới")
    trainer.train()

trainer.save_model(FINAL_SAVE_PATH)
tokenizer.save_pretrained(FINAL_SAVE_PATH)

# PHẦN 4: INFERENCE
print("\n>>> [4/5] Chạy Inference...")
model.eval()
predictions, gold_lines = [], []

for item in val_ds:
    input_text = f"translate to SQL: {item['question']} | Schema: {get_crp_schema(item['db_id'])}"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        output = model.generate(input_ids, max_length=128, num_beams=4, early_stopping=True)
    predictions.append(tokenizer.decode(output[0], skip_special_tokens=True) + "\n")
    gold_lines.append(f"{item['query']}\t{item['db_id']}\n")

with open('pred.txt', 'w') as f: f.writelines(predictions)
with open('gold.txt', 'w') as f: f.writelines(gold_lines)

print("\n>>> [5/5] Kết quả đánh giá:")
!sed -i 's/conn = sqlite3.connect(db)/conn = sqlite3.connect(db)\n    conn.text_factory = lambda b: b.decode(errors="ignore")/' evaluation.py
!python evaluation.py --gold gold.txt --pred pred.txt --db spider_data/database --table spider_data/tables.json --etype all

/tmp/ipython-input-3984788200.py:26: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.multiarray.
  np.core.multiarray._reconstruct,


✅ Đã whitelist Numpy cho PyTorch.
✅ Đã ép buộc torch.load(weights_only=False) thành công.
Mounted at /content/drive
>>> [1/5] Tải dữ liệu...
Dataset URL: https://www.kaggle.com/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset
License(s): unknown
  0% 0.00/96.0M [00:00<?, ?B/s]
100% 96.0M/96.0M [00:00<00:00, 1.33GB/s]


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


>>> [4/5] Chạy Inference...


Token indices sequence length is longer than the specified maximum sequence length for this model (874 > 512). Running this sequence through the model will result in indexing errors



>>> [5/5] Kết quả đánh giá:
eval_err_num:1
medium pred: SELECT T1.name, T1.year FROM singer AS T1 JOIN song_in_concert AS T2 ON T1.singer_id = T2.singer_id JOIN concert AS T3 ON T2.concert_id = T3.concert_id ORDER BY T3.age DESC LIMIT 1
medium gold: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

eval_err_num:2
medium pred: SELECT T1.name, T1.year FROM singer AS T1 JOIN song_release_year AS T2 ON T1.singer_id = T2.singer_id JOIN concert AS T3 ON T2.concert_id = T3.concert_id WHERE T3.age = (SELECT max(age) FROM singer AS T4 JOIN singer_in_concert AS T4 ON T3.concert_id = T4.conce
medium gold: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

hard pred: SELECT name FROM singer WHERE age > (SELECT avg(age) FROM singer)
hard gold: SELECT song_name FROM singer WHERE age  >  (SELECT avg(age) FROM singer)

hard pred: SELECT name FROM singer WHERE age > (SELECT avg(age) FROM singer)
hard gold: SELECT song_name FROM singer WHERE age  >  (SELECT av

In [2]:

# Efficiency Evaluation

# 1. Import libraries
import torch
import time
import numpy as np
import psutil
import os
from transformers import T5Tokenizer, T5ForConditionalGeneration

# 2. Định nghĩa đường dẫn và thiết bị
# Phải khớp với thư mục bạn đã lưu ở code train
FINAL_SAVE_PATH = "/content/drive/My Drive/T5_Small_Spider_CRP_FFT"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Đang tải mô hình từ: {FINAL_SAVE_PATH}")
print(f"Sử dụng thiết bị: {device}")

# 3. Tải mô hình và Tokenizer
try:
    tokenizer = T5Tokenizer.from_pretrained(FINAL_SAVE_PATH)
    model = T5ForConditionalGeneration.from_pretrained(FINAL_SAVE_PATH)
    model.to(device)
    model.eval()
    print("✅ Tải mô hình thành công!")
except Exception as e:
    print(f"❌ Lỗi khi tải mô hình: {e}")
    print("Vui lòng kiểm tra lại xem đường dẫn FINAL_SAVE_PATH có tồn tại mô hình chưa.")
    exit()

# 4. Chuẩn bị dữ liệu đầu vào mẫu
sample_question = "How many students are there?"
sample_schema = "CREATE TABLE student(id int, name text);"
input_text = f"translate to SQL: {sample_question} | Schema: {sample_schema}"

inputs = tokenizer(
    input_text,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(device)

# 5. Khởi động GPU
print("Đang warmup GPU...")
for _ in range(10):
    with torch.no_grad():
        model.generate(**inputs, max_length=128)

# 6. Đo độ trễ suy luận (Inference Latency)
print("Đang đo lường độ trễ (100 lần chạy)...")
latencies = []

for _ in range(100):
    start = time.time()

    with torch.no_grad():
        model.generate(**inputs, max_length=128)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    latencies.append(time.time() - start)

avg_latency = np.mean(latencies)
std_latency = np.std(latencies)


throughput = 1 / avg_latency

# 8. Đo VRAM sử dụng
gpu_memory_allocated = None
gpu_memory_reserved = None

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        model.generate(**inputs, max_length=128)

    gpu_memory_allocated = torch.cuda.max_memory_allocated() / 1024**2
    gpu_memory_reserved = torch.cuda.max_memory_reserved() / 1024**2

# 9. CPU RAM usage
process = psutil.Process(os.getpid())
ram_usage = process.memory_info().rss / 1024**2

# 10. Số lượng tham số của mô hình
total_params = sum(p.numel() for p in model.parameters())

# 11. In kết quả
print("\n" + "="*40)
print("     ĐÁNH GIÁ HIỆU NĂNG (T5-SMALL)      ")
print("="*40)
print(f"Độ trễ TB (Inference Latency):  {avg_latency*1000:.2f} ms")
print(f"Độ lệch chuẩn Latency:          {std_latency*1000:.2f} ms")
print(f"Thông lượng (Throughput - BS=1):{throughput:.2f} samples/sec")

if torch.cuda.is_available():
    print(f"VRAM cấp phát cho Tensor:       {gpu_memory_allocated:.2f} MB")
    print(f"VRAM thực tế PyTorch giữ:       {gpu_memory_reserved:.2f} MB")
else:
    print("VRAM (GPU Memory Usage):        Không khả dụng (Đang chạy CPU)")

print(f"RAM CPU sử dụng:                {ram_usage:.2f} MB")
print(f"Tổng số tham số mô hình:        {total_params:,}")
print("="*40)

Đang tải mô hình từ: /content/drive/My Drive/T5_Small_Spider_CRP_FFT
Sử dụng thiết bị: cuda


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

✅ Tải mô hình thành công!
Đang warmup GPU...
Đang đo lường độ trễ (100 lần chạy)...

     ĐÁNH GIÁ HIỆU NĂNG (T5-SMALL)      
Độ trễ TB (Inference Latency):  114.11 ms
Độ lệch chuẩn Latency:          19.12 ms
Thông lượng (Throughput - BS=1):8.76 samples/sec
VRAM cấp phát cho Tensor:       844.92 MB
VRAM thực tế PyTorch giữ:       856.00 MB
RAM CPU sử dụng:                1680.54 MB
Tổng số tham số mô hình:        60,506,624
